<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/16-time-series-and-recommendations/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 16: Exercises

Twelve exercises: four Easy, five Medium and three Hard. The first half builds forecasting tools: lag and leak-free rolling features, the seasonal naive baseline, MASE, time-ordered validation splits, exponential smoothing, recursive multi-step forecasts and Holt's method. The second half builds a recommender: ranking metrics, item similarities, matrix factorisation by alternating least squares, and the final top-k ranking with sensible tie-breaking.

Several checks compare your code with pandas or scikit-learn, and the others with a reference implementation of exactly the procedure described.

**Run the setup cell first.**

Rules:

- NumPy and pandas. Don't call scikit-learn's version of what you're writing (such as `TimeSeriesSplit`); the checks use it to test your answer.
- Solutions are in `03_solutions.ipynb`.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import TimeSeriesSplit


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _series(n=60, seed=0):
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2024-01-01", periods=n, freq="D")
    return pd.Series(100 + np.arange(n) + 10 * np.sin(np.arange(n) * 2 * np.pi / 7) + rng.normal(0, 3, n), index=idx)


def check_lag_features(f):
    s = _series()
    expected = pd.DataFrame({"lag_1": s.shift(1), "lag_7": s.shift(7), "lag_14": s.shift(14)}, index=s.index)
    _scenarios("lag_features", [
        ("the example", lambda: f(pd.Series([10.0, 20, 30, 40]), [1, 2]),
         pd.DataFrame({"lag_1": [np.nan, 10, 20, 30], "lag_2": [np.nan, np.nan, 10, 20]}), "shift(k) looks k steps into the past"),
        ("a dated series keeps its index", lambda: f(s, [1, 7, 14]), expected, None),
        ("columns in the order given", lambda: list(f(s, [7, 1]).columns), ["lag_7", "lag_1"], None),
        ("lag 0 is the series itself", lambda: f(s, [0])["lag_0"].tolist(), s.tolist(), None),
    ])


def check_rolling_features(f):
    s = _series()
    past = s.shift(1).rolling(7)
    expected = pd.DataFrame({"mean_7": past.mean(), "min_7": past.min(), "max_7": past.max()}, index=s.index)
    spike = pd.Series([1.0, 1, 1, 1, 100])
    _scenarios("rolling_features", [
        ("the example", lambda: f(pd.Series([1.0, 2, 3, 4, 5]), 2)["mean_2"].to_numpy(),
         np.array([np.nan, np.nan, 1.5, 2.5, 3.5]), "shift(1) before rolling"),
        ("mean, min and max of the previous 7 days", lambda: f(s, 7), expected, None),
        ("the current value is never included", lambda: float(f(spike, 3)["max_3"].iloc[-1]), 1.0,
         "the window must end one step before t"),
        ("NaN until a full window exists", lambda: int(f(s, 7)["mean_7"].isna().sum()), 7, None),
    ])


def check_seasonal_naive(f):
    _scenarios("seasonal_naive", [
        ("the example", lambda: f([1, 2, 3, 10, 20, 30], 3, 5), np.array([10.0, 20, 30, 10, 20]), "repeat the last full season"),
        ("season 1 is the naive forecast", lambda: f([5, 6, 7], 1, 4), np.array([7.0, 7, 7, 7]), None),
        ("a horizon shorter than a season", lambda: f(np.arange(14), 7, 3), np.array([7.0, 8, 9]), None),
        ("returns floats", lambda: f([1, 2], 1, 2).dtype.kind, "f", None),
        ("too little history raises", lambda: f([1, 2], 7, 3), Raises(ValueError), None),
    ])


def check_precision_recall_at_k(f):
    _scenarios("precision_recall_at_k", [
        ("a small example", lambda: f(["a", "b", "c", "d"], {"b", "d", "x"}, 3), (1 / 3, 1 / 3),
         "only the first k recommendations count"),
        ("all hits", lambda: f([1, 2], [1, 2], 2), (1.0, 1.0), None),
        ("a short list is penalised", lambda: f([1], [1, 2], 5), (0.2, 0.5), "divide by k, not by the list length"),
        ("relevant can be any collection", lambda: f(np.array([3, 4, 5]), np.array([5, 9]), 3), (1 / 3, 0.5), None),
        ("returns floats", lambda: [type(v).__name__ for v in f([1], [1], 1)], ["float", "float"], None),
        ("no relevant items raises", lambda: f([1, 2], [], 2), Raises(ValueError), None),
    ])


def check_forecast_errors(f):
    rng = np.random.default_rng(1)
    train = 50 + 10 * np.sin(np.arange(140) * 2 * np.pi / 7) + rng.normal(0, 2, 140)
    actual = 50 + 10 * np.sin(np.arange(140, 168) * 2 * np.pi / 7) + rng.normal(0, 2, 28)
    forecast = np.resize(train[-7:], 28)
    e = actual - forecast
    _scenarios("forecast_errors", [
        ("a small example", lambda: f([3, 5], [1, 5], [1, 2, 4, 7]), {"MAE": 1.0, "RMSE": math.sqrt(2), "MASE": 0.5},
         "naive training errors: |2-1|, |4-2|, |7-4|, mean 2"),
        ("a weekly series with season_length 7", lambda: f(actual, forecast, train, 7),
         {"MAE": float(np.mean(np.abs(e))), "RMSE": float(np.sqrt(np.mean(e ** 2))),
          "MASE": float(np.mean(np.abs(e)) / np.mean(np.abs(train[7:] - train[:-7])))}, "compare train[t] with train[t - season_length]"),
        ("the right season beats the wrong one", lambda: f(actual, forecast, train, 7)["MASE"] > f(actual, forecast, train, 1)["MASE"], True,
         "a weekly series is predicted much better by last week than by yesterday, so its scale is smaller"),
        ("different lengths raise", lambda: f([1, 2, 3], [1, 2], [1, 2, 3]), Raises(ValueError), None),
        ("one forecast for three actual values raises", lambda: f([1, 2, 3], [2], [1, 2, 3]), Raises(ValueError),
         "NumPy would broadcast a single forecast, so compare the lengths yourself"),
        ("a constant training series raises", lambda: f([1, 2], [1, 2], [5, 5, 5]), Raises(ValueError), None),
    ])


def check_expanding_window_splits(f):
    def ref(n, n_splits, test_size, gap=0):
        return [(tr, te) for tr, te in TimeSeriesSplit(n_splits=n_splits, test_size=test_size, gap=gap).split(np.zeros(n))]

    def as_lists(splits):
        return [(np.asarray(a).tolist(), np.asarray(b).tolist()) for a, b in splits]

    def all_before():
        return all(np.max(tr) < np.min(te) for tr, te in f(500, 6, 30, gap=5))

    _scenarios("expanding_window_splits", [
        ("a small example", lambda: as_lists(f(10, 3, 2)), [(list(range(0, 4)), [4, 5]), (list(range(0, 6)), [6, 7]), (list(range(0, 8)), [8, 9])],
         "test blocks cover the end of the series"),
        ("matches TimeSeriesSplit", lambda: as_lists(f(731, 5, 60)), as_lists(ref(731, 5, 60)), None),
        ("with a gap", lambda: as_lists(f(731, 4, 30, gap=7)), as_lists(ref(731, 4, 30, 7)), "leave out gap indices before each test block"),
        ("training always comes before testing", all_before, True, None),
        ("too many splits raise", lambda: f(10, 5, 2), Raises(ValueError), None),
    ])


def check_exponential_smoothing(f):
    s = _series(80, seed=3)
    ref_level = s.ewm(alpha=0.3, adjust=False).mean()
    _scenarios("exponential_smoothing", [
        ("a small example", lambda: f([10, 20, 30], 0.5), (np.array([np.nan, 10.0, 15.0]), 22.5),
         "forecast[t] is the level before seeing y[t]"),
        ("matches pandas ewm", lambda: f(s, 0.3), (ref_level.shift(1).to_numpy(), float(ref_level.iloc[-1])), None),
        ("alpha 1 is the naive forecast", lambda: f([3, 8, 1, 4], 1.0), (np.array([np.nan, 3.0, 8.0, 1.0]), 4.0), None),
        ("alpha 0 raises", lambda: f([1, 2], 0), Raises(ValueError), None),
    ])


def check_recursive_forecast(f):
    class OnlyLag1:
        def predict(self, X):
            X = np.asarray(X)
            assert X.ndim == 2 and X.shape[0] == 1, "predict should get a 2D array with one row"
            return X[:, 0] * 2

    class Average:
        def predict(self, X):
            return np.asarray(X).mean(axis=1)

    rng = np.random.default_rng(4)
    y = np.zeros(300)
    for t in range(2, 300):
        y[t] = 0.6 * y[t - 1] - 0.3 * y[t - 2] + rng.normal()
    lags = np.column_stack([y[1:-1], y[:-2]])
    ar = LinearRegression().fit(lags, y[2:])

    def manual_ar(history, horizon):
        h = list(history)
        out = []
        for _ in range(horizon):
            p = ar.intercept_ + ar.coef_[0] * h[-1] + ar.coef_[1] * h[-2]
            out.append(p)
            h.append(p)
        return np.array(out)

    _scenarios("recursive_forecast", [
        ("the most recent value comes first", lambda: f(OnlyLag1(), [1, 2, 3], 3, 3), np.array([6.0, 12.0, 24.0]),
         "row = [y[t-1], y[t-2], ...]: reverse the last n_lags values"),
        ("forecasts are fed back in", lambda: f(Average(), [0, 0, 6], 3, 3), np.array([2.0, 8 / 3, 32 / 9]), None),
        ("matches a fitted autoregressive model", lambda: f(ar, y[:250], 2, 10), manual_ar(y[:250], 10), None),
        ("too little history raises", lambda: f(Average(), [1], 3, 2), Raises(ValueError), None),
    ])


def check_item_cosine_similarity(f):
    rng = np.random.default_rng(5)
    R = (rng.random((50, 12)) < 0.2).astype(float)
    R[:, 3] = 0                                         # an item nobody bought
    ref = cosine_similarity(R.T)
    np.fill_diagonal(ref, 0)
    ratings = rng.integers(0, 6, (40, 8)).astype(float)
    ref2 = cosine_similarity(ratings.T)
    np.fill_diagonal(ref2, 0)
    _scenarios("item_cosine_similarity", [
        ("a small example", lambda: f(np.array([[1, 1, 0], [1, 0, 1], [0, 0, 1]])),
         np.array([[0, 1 / math.sqrt(2), 0.5], [1 / math.sqrt(2), 0, 0], [0.5, 0, 0]]), "dot products divided by the column lengths"),
        ("matches scikit-learn on purchases, with an empty item", lambda: f(R), ref, "an item with no entries gets similarity 0"),
        ("matches scikit-learn on ratings", lambda: f(ratings), ref2, None),
        ("the diagonal is 0", lambda: float(np.abs(np.diag(f(ratings))).max()), 0.0, None),
    ])


def check_holt_forecast(f):
    rng = np.random.default_rng(6)
    y = np.cumsum(rng.normal(1, 2, 60)) + 50

    def ref(series, a, b, h):
        level, trend = series[0], series[1] - series[0]
        for t in range(1, len(series)):
            new = a * series[t] + (1 - a) * (level + trend)
            trend = b * (new - level) + (1 - b) * trend
            level = new
        return level + trend * np.arange(1, h + 1)

    bend = np.r_[np.arange(10.0), 10 + 3 * np.arange(1, 11)]
    _scenarios("holt_forecast", [
        ("a straight line continues exactly", lambda: f(np.arange(5, 25, 2.0), 0.3, 0.2, 3), np.array([25.0, 27.0, 29.0]), None),
        ("matches the reference", lambda: f(y, 0.5, 0.3, 6), ref(y, 0.5, 0.3, 6), "update the trend with the change from the old level to the new one"),
        ("a series whose trend changes", lambda: f(bend, 0.8, 0.5, 4), ref(bend, 0.8, 0.5, 4), None),
        ("forecasts can go beyond the history", lambda: bool(f(np.arange(10.0), 0.5, 0.5, 5).max() > 9), True, None),
        ("one value raises", lambda: f([3.0], 0.5, 0.5, 2), Raises(ValueError), None),
        ("beta of 0 raises", lambda: f([1.0, 2.0], 0.5, 0, 2), Raises(ValueError), None),
    ])


def check_als_factorise(f):
    rng = np.random.default_rng(7)
    n_u, n_i, k = 40, 30, 2
    P_true, Q_true = rng.normal(size=(n_u, k)), rng.normal(size=(n_i, k))
    mask = rng.random((n_u, n_i)) < 0.4
    mask[5] = False                                       # a user with no entries
    users, items = np.nonzero(mask)
    values = np.sum(P_true[users] * Q_true[items], axis=1) + rng.normal(0, 0.1, len(users))

    def ref(n_factors, reg, n_iter, seed=0):
        g = np.random.default_rng(seed)
        P = g.normal(0, 0.1, size=(n_u, n_factors))
        Q = g.normal(0, 0.1, size=(n_i, n_factors))
        for _ in range(n_iter):
            for u in range(n_u):
                r = np.flatnonzero(users == u)
                P[u] = 0.0 if len(r) == 0 else np.linalg.solve(Q[items[r]].T @ Q[items[r]] + reg * np.eye(n_factors), Q[items[r]].T @ values[r])
            for i in range(n_i):
                r = np.flatnonzero(items == i)
                Q[i] = 0.0 if len(r) == 0 else np.linalg.solve(P[users[r]].T @ P[users[r]] + reg * np.eye(n_factors), P[users[r]].T @ values[r])
        return P, Q

    def objective(P, Q, reg):
        return float(np.sum((values - np.sum(P[users] * Q[items], axis=1)) ** 2) + reg * (np.sum(P ** 2) + np.sum(Q ** 2)))

    def fits_well():
        P, Q = f(users, items, values, n_u, n_i, 2, 1.0, 20)
        return bool(np.sqrt(np.mean((values - np.sum(P[users] * Q[items], axis=1)) ** 2)) < 0.2)

    def never_increases():
        objs = [objective(*f(users, items, values, n_u, n_i, 2, 0.5, it), 0.5) for it in range(1, 7)]
        return bool(all(b <= a + 1e-9 for a, b in zip(objs, objs[1:])))

    _scenarios("als_factorise", [
        ("matches the reference procedure", lambda: f(users, items, values, n_u, n_i, 2, 0.5, 5), ref(2, 0.5, 5),
         "one rng: P first, then Q; users in order, then items"),
        ("another setting", lambda: f(users, items, values, n_u, n_i, 3, 2.0, 3, seed=4), ref(3, 2.0, 3, seed=4), None),
        ("a user with no entries gets zeros", lambda: f(users, items, values, n_u, n_i, 2, 0.5, 2)[0][5].tolist(), [0.0, 0.0], None),
        ("the regularised error never increases", never_increases, True, None),
        ("fits the known entries well", fits_well, True, None),
    ])


def check_recommend_top_k(f):
    scores = np.array([[0.9, 0.1, 0.5, 0.5, 0.0],
                       [0.0, 0.0, 0.0, 0.0, 0.0],
                       [1.0, 2.0, 3.0, 4.0, 5.0]])
    seen = np.array([[True, False, False, False, False],
                     [False, False, False, False, False],
                     [False, True, True, True, True]])
    popularity = np.array([10, 50, 5, 20, 30])
    rng = np.random.default_rng(8)
    S = np.round(rng.random((20, 40)), 1)
    seen_big = rng.random((20, 40)) < 0.3
    pop_big = rng.integers(0, 100, 40)

    def ref_lists():
        out = []
        for u in range(20):
            cand = [i for i in range(40) if not seen_big[u, i]]
            cand.sort(key=lambda i: (-S[u, i], -pop_big[i], i))
            out.append(cand[:7])
        return out

    _scenarios("recommend_top_k", [
        ("seen items are skipped; ties go to the more popular item", lambda: f(scores, seen, popularity, 3)[0].tolist(), [3, 2, 1],
         "items 2 and 3 tie on score; item 3 is more popular"),
        ("a user with no signal gets the most popular items", lambda: f(scores, seen, popularity, 3)[1].tolist(), [1, 4, 3], None),
        ("fewer unseen items than k: return them all", lambda: f(scores, seen, popularity, 3)[2].tolist(), [0], None),
        ("matches a reference ranking", lambda: [r.tolist() for r in f(S, seen_big, pop_big, 7)], ref_lists(), None),
        ("one array per user", lambda: len(f(S, seen_big, pop_big, 5)), 20, None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `lag_features`

Write `lag_features(series, lags)` that turns a pandas Series into a DataFrame of lagged copies, for forecasting.

- One column per lag, named `"lag_1"`, `"lag_7"` and so on, in the order given.
- Column `lag_k` holds the value `k` steps **earlier**, so the first `k` rows are `NaN`.
- Keep the series' index.

Example: for the series `[10, 20, 30, 40]`, `lag_features(s, [1, 2])["lag_1"]` is `[NaN, 10, 20, 30]`.

In [2]:
def lag_features(series: pd.Series, lags) -> pd.DataFrame:
    # your code here
    pass

In [3]:
check_lag_features(lag_features)

FAIL  lag_features: the example
      expected: DataFrame shape=(4, 2)
             lag_1  lag_2
          0    NaN    NaN
          1   10.0    NaN
          2   20.0   10.0
          3   30.0   20.0
      got:      None
      hint: shift(k) looks k steps into the past


### Exercise 2: `rolling_features`

Write `rolling_features(series, window)` that computes leak-free rolling statistics: for each time `t`, the **mean**, **min** and **max** of the `window` values **before** `t` (not including `t` itself).

- Return a DataFrame with columns `f"mean_{window}"`, `f"min_{window}"` and `f"max_{window}"`, with the series' index.
- A row has `NaN` until a full `window` of earlier values exists.

Example: for `[1, 2, 3, 4, 5]` and `window=2`, the means are `[NaN, NaN, 1.5, 2.5, 3.5]`.

In [4]:
def rolling_features(series: pd.Series, window: int) -> pd.DataFrame:
    # your code here
    pass

In [5]:
check_rolling_features(rolling_features)

FAIL  rolling_features: the example
      raised TypeError: 'NoneType' object is not subscriptable
      expected: array(shape=(5,), dtype=float64, [nan, nan, 1.5, 2.5, 3.5])
      hint: shift(1) before rolling


### Exercise 3: `seasonal_naive`

Write `seasonal_naive(history, season_length, horizon)` that forecasts the next `horizon` values by repeating the **last full season** of `history`.

- With `season_length=7` on daily data, the forecast for each of the next days is the value from the same weekday of the last week, repeated as many times as needed.
- With `season_length=1` this is the naive forecast (the last value, repeated).
- Return a float NumPy array. Raise `ValueError` if `history` has fewer than `season_length` values.

Example: `seasonal_naive([1, 2, 3, 10, 20, 30], 3, 5)` returns `[10, 20, 30, 10, 20]`.

In [6]:
def seasonal_naive(history, season_length: int, horizon: int) -> np.ndarray:
    # your code here
    pass

In [7]:
check_seasonal_naive(seasonal_naive)

FAIL  seasonal_naive: the example
      expected: array(shape=(5,), dtype=float64, [10., 20., 30., 10., 20.])
      got:      None
      hint: repeat the last full season


### Exercise 4: `precision_recall_at_k`

Write `precision_recall_at_k(recommended, relevant, k)` for evaluating one user's recommendations. Return `(precision, recall)` as two floats:

- `recommended`: the recommended items, best first (a list with no repeats). Only the first `k` count.
- `relevant`: the items the user actually liked or bought (any collection).
- precision = hits / `k`; recall = hits / number of relevant items, where hits = how many of the first `k` recommended items are relevant.

Divide by `k` even if fewer than `k` items were recommended (a short list is penalised). Raise `ValueError` if `relevant` is empty, since recall is undefined.

In [8]:
def precision_recall_at_k(recommended, relevant, k: int):
    # your code here
    pass

In [9]:
check_precision_recall_at_k(precision_recall_at_k)

FAIL  precision_recall_at_k: a small example
      expected: (0.3333333333333333, 0.3333333333333333)
      got:      None
      hint: only the first k recommendations count


## Medium

### Exercise 5: `forecast_errors`

Write `forecast_errors(actual, forecast, train, season_length=1)` that returns a dict of three floats:

- `"MAE"`: mean absolute error
- `"RMSE"`: root mean squared error
- `"MASE"`: the MAE divided by the MAE of the seasonal naive forecast **on the training data**: the mean of `|train[t] - train[t - season_length]|` over all `t` where both exist

MASE below 1 means the forecast beats the (in-sample) seasonal naive forecast. Raise `ValueError` if `actual` and `forecast` have different lengths, or if the training scale is 0.

In [10]:
def forecast_errors(actual, forecast, train, season_length: int = 1) -> dict:
    # your code here
    pass

In [11]:
check_forecast_errors(forecast_errors)

FAIL  forecast_errors: a small example
      expected: {'MAE': 1.0, 'RMSE': 1.4142135623730951, 'MASE': 0.5}
      got:      None
      hint: naive training errors: |2-1|, |4-2|, |7-4|, mean 2


### Exercise 6: `expanding_window_splits`

Write `expanding_window_splits(n_samples, n_splits, test_size, gap=0)` that returns a list of `(train_indices, test_indices)` pairs of integer arrays for time-ordered validation:

- The `n_splits` test blocks each have `test_size` consecutive indices and together cover the **last** `n_splits × test_size` indices, in order.
- Each training set is every index from 0 up to the start of its test block, minus the `gap` indices just before the test block.
- Raise `ValueError` if any training set would be empty.

This matches `sklearn.model_selection.TimeSeriesSplit(n_splits, test_size=test_size, gap=gap)`, which the check compares with. Don't use it yourself.

In [12]:
def expanding_window_splits(n_samples: int, n_splits: int, test_size: int, gap: int = 0):
    # your code here
    pass

In [13]:
check_expanding_window_splits(expanding_window_splits)

FAIL  expanding_window_splits: a small example
      raised TypeError: 'NoneType' object is not iterable
      expected: [([0, 1, 2, 3], [4, 5]), ([0, 1, 2, 3, 4, 5], [6, 7]), ([0, 1, 2, 3, 4, 5, 6, 7], [8, 9])]
      hint: test blocks cover the end of the series


### Exercise 7: `exponential_smoothing`

Write `exponential_smoothing(y, alpha)` for simple exponential smoothing. Return `(forecasts, next_forecast)`:

- Start with `level = y[0]`. For each `t` from 1 on, the one-step forecast for `y[t]` is the current level; then update `level = alpha * y[t] + (1 - alpha) * level`.
- `forecasts` is a float array the same length as `y`, with `forecasts[0] = np.nan` (no forecast for the first value) and `forecasts[t]` the forecast made before seeing `y[t]`.
- `next_forecast` is the final level: the forecast for the next, unseen value, as a float.
- Raise `ValueError` unless `0 < alpha <= 1`.

In [14]:
def exponential_smoothing(y, alpha: float):
    # your code here
    pass

In [15]:
check_exponential_smoothing(exponential_smoothing)

FAIL  exponential_smoothing: a small example
      expected: (array(shape=(3,), dtype=float64, [nan, 10., 15.]), 22.5)
      got:      None
      hint: forecast[t] is the level before seeing y[t]


### Exercise 8: `recursive_forecast`

Write `recursive_forecast(model, history, n_lags, horizon)` that forecasts several steps ahead with a one-step model.

- `model` has a `predict(X)` method, where each row of `X` holds the last `n_lags` values **most recent first**: `[y[t-1], y[t-2], ..., y[t-n_lags]]`.
- Forecast one step, append the forecast to the history as if it were real, forecast the next step from the updated history, and so on, `horizon` times.
- Return the `horizon` forecasts as a float array. Call `model.predict` once per step, with a 2D array of one row.
- Raise `ValueError` if `history` has fewer than `n_lags` values.

In [16]:
def recursive_forecast(model, history, n_lags: int, horizon: int) -> np.ndarray:
    # your code here
    pass

In [17]:
check_recursive_forecast(recursive_forecast)

FAIL  recursive_forecast: the most recent value comes first
      expected: array(shape=(3,), dtype=float64, [ 6., 12., 24.])
      got:      None
      hint: row = [y[t-1], y[t-2], ...]: reverse the last n_lags values


### Exercise 9: `item_cosine_similarity`

Write `item_cosine_similarity(R)` that computes the cosine similarity between every pair of **items** (columns) of a users × items matrix `R` (a NumPy array, with 0 meaning "no rating" or "not bought").

- Return an items × items float array.
- Set the diagonal to 0 (an item isn't recommended because of itself).
- An item with no non-zero entries has similarity 0 with everything (no division by zero).

Use matrix operations, not a loop over pairs of items.

In [18]:
def item_cosine_similarity(R) -> np.ndarray:
    # your code here
    pass

In [19]:
check_item_cosine_similarity(item_cosine_similarity)

FAIL  item_cosine_similarity: a small example
      expected: array(shape=(3, 3), dtype=float64, [[0. , 0.7071, 0.5 ], [0.7071, 0. , 0. ], [0.5 , 0. ...
      got:      None
      hint: dot products divided by the column lengths


## Hard

### Exercise 10: `holt_forecast`

Write `holt_forecast(y, alpha, beta, horizon)` for **Holt's linear method**: exponential smoothing with a trend. Return the forecasts for the next `horizon` steps as a float array.

- Start with `level = y[0]` and `trend = y[1] - y[0]`.
- For each `t` from 1 to `n - 1`:
  - `new_level = alpha * y[t] + (1 - alpha) * (level + trend)`
  - `trend = beta * (new_level - level) + (1 - beta) * trend`
  - `level = new_level`
- The forecast `h` steps ahead (h = 1, 2, ..., horizon) is `level + h * trend`.
- Raise `ValueError` if `y` has fewer than 2 values, or if `alpha` or `beta` isn't in `(0, 1]`.

In [20]:
def holt_forecast(y, alpha: float, beta: float, horizon: int) -> np.ndarray:
    # your code here
    pass

In [21]:
check_holt_forecast(holt_forecast)

FAIL  holt_forecast: a straight line continues exactly
      expected: array(shape=(3,), dtype=float64, [25., 27., 29.])
      got:      None


### Exercise 11: `als_factorise`

Write `als_factorise(users, items, values, n_users, n_items, n_factors, reg, n_iter, seed=0)` that fits a matrix factorisation by **alternating least squares**, so that `values[j] ≈ P[users[j]] · Q[items[j]]`. Return `(P, Q)` with shapes `(n_users, n_factors)` and `(n_items, n_factors)`.

`users`, `items` and `values` are equal-length arrays describing the known entries. Follow this procedure exactly:

```
rng = np.random.default_rng(seed)
P = rng.normal(0, 0.1, size=(n_users, n_factors))
Q = rng.normal(0, 0.1, size=(n_items, n_factors))
repeat n_iter times:
    for each user u = 0, 1, ..., n_users - 1:
        if u has no entries: P[u] = 0
        else: P[u] = solve(Qu.T @ Qu + reg * I, Qu.T @ v_u)     # Qu: rows of Q for u's items, v_u: u's values
    then the same for each item i, using the updated P
```

In [22]:
def als_factorise(users, items, values, n_users: int, n_items: int, n_factors: int, reg: float, n_iter: int, seed: int = 0):
    # your code here
    pass

In [23]:
check_als_factorise(als_factorise)

FAIL  als_factorise: matches the reference procedure
      expected: (array(shape=(40, 2), dtype=float64, [[ 0.0895, -0.2066], [ 0.366 , -0.0176], [-0.6905, ..., array(shape=(30, 2), dtype=float64, [[-0.2919, -0.3172], [ 0.3264, 0.139 ], [-0.7357, 1...)
      got:      None
      hint: one rng: P first, then Q; users in order, then items


### Exercise 12: `recommend_top_k`

Write `recommend_top_k(scores, seen, popularity, k)` that turns a users × items score matrix into top-k recommendation lists. Return a list with one integer array per user.

- Never recommend an item the user has already seen (`seen` is a boolean users × items array).
- Rank the remaining items by score, highest first. Break ties by **popularity** (higher first, `popularity` is one number per item), then by **item index** (lower first).
- If a user has fewer than `k` unseen items, return all of them, ranked.

This tie-breaking makes popularity the automatic fallback: a new user whose scores are all 0 gets the most popular items.

In [24]:
def recommend_top_k(scores, seen, popularity, k: int) -> list:
    # your code here
    pass

In [25]:
check_recommend_top_k(recommend_top_k)

FAIL  recommend_top_k: seen items are skipped; ties go to the more popular item
      raised TypeError: 'NoneType' object is not subscriptable
      expected: [3, 2, 1]
      hint: items 2 and 3 tie on score; item 3 is more popular


## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the mini project in `04_mini_project.ipynb`, which forecasts hourly bike demand and recommends products to the customers of an online shop, both tested on the future.